# Modul 4: Regularisasi dan Hyperparameter Tuning

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  

Simpan berkas ini sebagai `M04_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Protokol tetap: subset $3\,000$/$3\,000$, baseline $784 \rightarrow 512 \rightarrow 512 \rightarrow 10$, Adam, batch $128$, $30$ epoch.
3. Setiap run regularisasi hanya boleh mengubah **satu** faktor dari baseline.
4. **Test set dilarang disentuh** sampai bagian F. Jangan membuat `ds_uji` lebih awal.
5. Catat seluruh run ke `metrics.csv`, termasuk trial yang buruk.
6. Luaran: `M04_NIM.ipynb`, `M04_NIM.pdf`, `M04_NIM_metrics.csv`, dan grafik generalization gap.

In [ ]:
import copy
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import FashionMNIST

NIM = 'TODO'                     # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42   # seed data dan init
SEED_CARI = SEED + 1             # seed pengambilan sampel random search
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE),
       'seed': SEED, 'seed_cari': SEED_CARI})

## A. Pre-lab dan protokol anti-kebocoran - 10 poin

1. **Beda overfitting dan underfitting dilihat dari kurva:** TODO
2. **Mengapa dropout dimatikan saat evaluasi, dan perintah apa yang melakukannya:** TODO
3. **Apa yang ditambahkan weight decay pada fungsi objektif:** TODO
4. **Mengapa memilih konfigurasi memakai test set membuat angka test tidak sah:** TODO

**Protokol anti-kebocoran saya.** Tuliskan kapan test set boleh dipakai dan bagaimana Anda memastikannya:

TODO

## B. Data - bagian dari 15 poin

Subset kecil di latih supaya overfitting terlihat. `ds_uji` **belum** dibuat di sini.

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

latih_penuh = FashionMNIST(root=DATA_ROOT, train=True, download=False,
                           transform=transforms.ToTensor())
uji_resmi = FashionMNIST(root=DATA_ROOT, train=False, download=False,
                         transform=transforms.ToTensor())

X_penuh = latih_penuh.data.float().unsqueeze(1) / 255.0
y_penuh = latih_penuh.targets

# TODO 1: ambil 3.000 latih dan 3.000 validasi terstratifikasi,
#         memakai train_test_split dengan random_state=SEED.
idx_latih, idx_val = ...

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]

# TODO 2: hitung MEAN dan STD dari subset LATIH saja.
MEAN, STD = ..., ...
normalkan = lambda t: (t - MEAN) / STD

ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)

print(f'latih {len(ds_latih)}  validasi {len(ds_val)}')
assert (len(ds_latih), len(ds_val)) == (3_000, 3_000)
assert torch.bincount(y_latih).min().item() == 300, 'subset harus terstratifikasi'
print('split sesuai protokol; ds_uji sengaja belum dibuat')

In [ ]:
BATCH, EPOCH = 128, 30

def buat_model(hidden=512, dropout=0.0):
    """TODO 3: 784 -> hidden -> hidden -> 10 dengan ReLU.

    Sisipkan nn.Dropout(dropout) setelah setiap ReLU HANYA bila dropout > 0.
    Selalu panggil seed_everything(SEED) lebih dahulu.
    """
    raise NotImplementedError

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    """TODO 4: kembalikan (loss rata-rata, akurasi).

    Jangan lupa model.eval() agar dropout nonaktif.
    """
    raise NotImplementedError

n_param = sum(p.numel() for p in buat_model().parameters())
print('parameter baseline:', n_param)
assert n_param == 669_706, 'arsitektur baseline belum sesuai protokol'

In [ ]:
def jalankan(hidden=512, dropout=0.0, weight_decay=0.0, lr=1e-3,
             epoch=EPOCH, sabar=None, label='baseline'):
    """TODO 5: satu fungsi pelatihan untuk SELURUH run modul ini.

    Wajib:
      - Adam dengan lr dan weight_decay yang diberikan;
      - catat train/val loss dan accuracy setiap epoch ke dict `riwayat`;
      - simpan state_dict pada epoch dengan val_loss terendah;
      - bila `sabar` diberikan, hentikan setelah `sabar` epoch tanpa perbaikan;
      - SEBELUM mengembalikan, muat kembali bobot terbaik;
      - kembalikan (model, riwayat, catatan) dengan catatan memuat run_id,
        seed, hidden, dropout, weight_decay, learning_rate, parameter,
        epoch_terbaik, epoch_berhenti, train_loss, val_loss, val_acc, gap,
        runtime_s.
    """
    raise NotImplementedError

## C. Baseline yang overfit - 15 poin

Latih baseline tanpa regularisasi apa pun, lalu tampilkan empat kurva dalam satu figur.

In [ ]:
# TODO 6: jalankan baseline, cetak catatannya, dan gambar dua panel:
#         (kiri) train vs validation loss, (kanan) train vs validation accuracy.
#         Tandai epoch terbaik dengan garis vertikal.
raise NotImplementedError

**Bukti overfitting.** Isi dengan angka Anda sendiri:

- Epoch dengan validation loss terendah: TODO
- Validation loss pada epoch terbaik dan pada epoch ke-30: TODO
- Gap pada epoch terakhir: TODO
- Kesimpulan: TODO

Bila validation loss Anda tidak pernah naik, periksa kembali ukuran subset dan jumlah epoch sebelum lanjut.

## D. Tiga strategi terpisah - 20 poin

Satu run, satu perubahan. Jangan menggabung ketiganya di tahap ini.

In [ ]:
# TODO 7: jalankan tiga run — dropout=0.5, weight_decay=1e-3, dan sabar=5 —
#         masing-masing hanya mengubah satu faktor dari baseline.
#         Kumpulkan ke daftar `hasil` bersama catatan baseline.
hasil = []
raise NotImplementedError

df_awal = pd.DataFrame(hasil)
print(df_awal[['run_id', 'epoch_terbaik', 'epoch_berhenti', 'train_loss',
               'val_loss', 'val_acc', 'gap', 'runtime_s']].to_string(index=False))
assert len(df_awal) == 4, 'harus ada baseline + tiga strategi'

**Perbandingan.** Jawab dengan angka dari tabel di atas:

- Strategi yang paling menurunkan `gap`: TODO
- Strategi yang paling menurunkan `val_loss`: TODO
- Apakah keduanya strategi yang sama? Mengapa? TODO

## E. Ruang pencarian dan random search - 25 poin

Ruang berisi $3^4 = 81$ kombinasi; anggaran $12$ trial. Seed pencarian wajib dicatat agar hasil dapat diulang.

In [ ]:
ruang = {'hidden': [64, 128, 256],
         'dropout': [0.0, 0.2, 0.5],
         'weight_decay': [0.0, 1e-4, 1e-3],
         'learning_rate': [1e-4, 3e-4, 1e-3]}

# TODO 8: susun seluruh kombinasi, ambil 12 secara acak TANPA pengulangan
#         memakai np.random.default_rng(SEED_CARI), lalu jalankan tiap trial
#         dengan sabar=5. Simpan ke daftar `trial`.
trial = []
raise NotImplementedError

tabel = pd.DataFrame(trial).sort_values('val_loss')
print(tabel[['run_id', 'hidden', 'dropout', 'weight_decay', 'learning_rate',
             'parameter', 'epoch_terbaik', 'val_loss', 'val_acc', 'gap']]
      .to_string(index=False))
assert len(tabel) == 12, 'harus ada dua belas trial'
assert tabel['run_id'].nunique() == 12, 'run_id tiap trial harus unik'

**Bacaan tabel.** Jawab dengan angka:

- Tiga trial teratas dan jumlah parameternya: TODO
- Apakah trial dengan parameter terbanyak berada di puncak? TODO
- Hyperparameter yang tampak paling berpengaruh: TODO

## F. Evaluasi akhir - 15 poin

Barulah di sini `ds_uji` dibuat. Sebelum menjalankan sel ini, pastikan seluruh keputusan sudah terkunci.

In [ ]:
# TODO 9: ambil konfigurasi terbaik dari `tabel`, latih ulang dengan seed sama,
#         BARU buat ds_uji dari uji_resmi, lalu evaluasi SATU KALI.
#         Tampilkan juga confusion matrix dan sebutkan dua kelas paling sering tertukar.
raise NotImplementedError

In [ ]:
# TODO 10: gabungkan seluruh run ke satu metrics.csv.
semua = pd.concat([df_awal, tabel], ignore_index=True)
semua.insert(0, 'module', 'M04')
semua.insert(1, 'student_id', NIM)
semua.to_csv(f'M04_{NIM}_metrics.csv', index=False)
print(f'{len(semua)} baris tersimpan')
assert len(semua) >= 16, 'metrics.csv minimal berisi 4 run awal + 12 trial'

## G. Pertanyaan analisis - bagian dari 15 poin

1. Pada epoch keberapa baseline mulai overfit, dan apa buktinya? TODO
2. Strategi mana yang paling menurunkan gap, dan mana yang paling menurunkan validation loss? TODO
3. Apakah trial dengan parameter terbanyak menghasilkan validation loss terendah? TODO
4. Hyperparameter mana yang paling berpengaruh, dan apa keterbatasan kesimpulan itu? TODO
5. Berapa selisih validation loss dan test loss model akhir, dan apa yang boleh disimpulkan darinya? TODO

**Keterbatasan.** Modul ini memakai $3\,000$ citra latih dan hanya $12$ trial dari $81$ kombinasi. Sebutkan apa yang tidak boleh disimpulkan: TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed data, dan seed pencarian tercantum.
- [ ] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [ ] Split, stratifikasi, dan jumlah parameter lolos sel pemeriksaan.
- [ ] Baseline benar-benar memperlihatkan kenaikan validation loss.
- [ ] Tiga strategi diuji terpisah, bukan digabung.
- [ ] Dua belas trial tercatat, termasuk yang buruk.
- [ ] `ds_uji` hanya muncul pada bagian F dan dievaluasi satu kali.
- [ ] Notebook lolos *Restart Kernel and Run All*.